<a href="https://colab.research.google.com/github/bhaskarpondugala/NASSCOM_AI_FDP/blob/main/Day10/U23_Cloud_Storage_Databases_Part1_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# U23 · Cloud Storage & Databases (Part 1) — Hands-On Notebook

**Goal:** *feel* the ideas from the slides by running tiny experiments on small,
made-up ("synthetic") data. No cloud account or internet needed — everything
uses Python's standard library (`sqlite3`, `json`, `os`, `time`, `random`).

How to use this notebook:
1. Read the short explanation above each code cell.
2. Run the cell (Shift + Enter) and read its printed output.
3. Try the *Your turn* tweaks at the end of each section.

Concepts covered: object / block / file storage · storage tiers · data lake vs
warehouse · SQL · ACID · NoSQL families · replication · sharding · CAP ·
indexing · backups & durability.


## 0 · Setup

We import a few standard-library modules and create a scratch folder so any
files we make stay tidy and easy to delete later.


In [1]:
import os            # work with files and folders
import json          # turn Python objects into text and back (JSON)
import time          # measure how long things take
import random        # generate fake/synthetic data
import sqlite3        # a tiny built-in SQL database (no install needed)
import shutil         # copy/remove folders (used for snapshots/cleanup)

random.seed(42)      # fix the random seed so results are the same every run

WORK = "scratch"                     # name of our scratch working folder
shutil.rmtree(WORK, ignore_errors=True)  # delete it if it already exists
os.makedirs(WORK, exist_ok=True)     # create a fresh, empty scratch folder
print("Scratch folder ready at:", os.path.abspath(WORK))  # confirm location


Scratch folder ready at: /content/scratch


## 1 · Three shapes of storage: object · block · file

From the slides:
- **Object** — each file is an *object* (data + metadata + unique key) in a flat
  *bucket*; no real folders. Great for scale and static data.
- **Block** — a raw disk split into fixed-size *blocks*; lowest latency; what
  databases sit on.
- **File** — a familiar *folder/path* hierarchy many machines can share.

We'll model all three in plain Python so the differences become obvious.


In [2]:
# ---------- OBJECT STORAGE (flat bucket: key -> {data, metadata}) ----------
object_bucket = {}   # a dict acts like a bucket; the dict key is the object key

def put_object(key, data, content_type):      # store one object whole
    object_bucket[key] = {                     # the value bundles everything
        "data": data,                          # the actual bytes/text
        "metadata": {                          # metadata travels WITH the object
            "size": len(data),                 # size in characters
            "content_type": content_type,      # e.g. image/png, text/plain
        },
    }

put_object("photos/cat.txt", "a cute cat", "text/plain")   # 'folders' are fake:
put_object("photos/dog.txt", "a good dog", "text/plain")   # they are just key text

print("Bucket keys:", list(object_bucket.keys()))          # flat list of keys
print("One object:", object_bucket["photos/cat.txt"])      # data + metadata together
# Note: to change an object you replace the WHOLE thing (no in-place edits).


Bucket keys: ['photos/cat.txt', 'photos/dog.txt']
One object: {'data': 'a cute cat', 'metadata': {'size': 10, 'content_type': 'text/plain'}}


In [3]:
# ---------- BLOCK STORAGE (one disk = list of fixed-size blocks) ----------
BLOCK_SIZE = 4                       # each block holds 4 characters (toy size)
disk = ["" for _ in range(8)]        # a 'disk' = 8 empty fixed-size blocks

def write_blocks(text):              # split text into fixed blocks and store
    for i in range(0, len(text), BLOCK_SIZE):     # step through text 4 chars at a time
        block_index = i // BLOCK_SIZE             # which block number this piece is
        disk[block_index] = text[i:i+BLOCK_SIZE]  # write that 4-char chunk in place

write_blocks("HELLO-DATABASE")       # store this string across several blocks
print("Raw blocks on disk:", disk)   # see the fixed-size pieces
# Block storage allows IN-PLACE edits: change just one block, not the whole file.
disk[0] = "JELL"                     # overwrite only block 0
print("After editing block 0:", disk)


Raw blocks on disk: ['HELL', 'O-DA', 'TABA', 'SE', '', '', '', '']
After editing block 0: ['JELL', 'O-DA', 'TABA', 'SE', '', '', '', '']


In [4]:
# ---------- FILE STORAGE (real folders + paths, shared filesystem) ----------
file_root = os.path.join(WORK, "fileshare")        # our 'mounted' share folder
os.makedirs(os.path.join(file_root, "team"), exist_ok=True)  # real nested folder

path = os.path.join(file_root, "team", "notes.txt")  # a real hierarchical path
with open(path, "w") as f:           # open the file for writing
    f.write("shared project notes")  # write some content

print("File exists at path:", os.path.exists(path))   # real path, real file
print("Contents:", open(path).read())                 # read it back
# Many machines could 'mount' file_root and see the same folders/paths.


File exists at path: True
Contents: shared project notes


**Takeaway:** object = flat keys + metadata (replace whole), block = fixed
in-place pieces under a database, file = familiar shared folders.

*Your turn:* add a third object under `photos/` and print the bucket keys again.


## 2 · Storage tiers (hot → cool → cold → archive)

Idea: move data to cheaper tiers as it *cools* (is accessed less). We attach a
"last accessed" age to each object and a **lifecycle policy** decides its tier.


In [5]:
# Synthetic objects: name -> days since last access
objects_age = {                      # smaller age = accessed more recently
    "live_dashboard.csv": 0,         # touched today  -> should be HOT
    "last_month_report.csv": 20,     # a few weeks old -> COOL
    "q1_logs.csv": 120,              # a few months old -> COLD
    "2019_taxes.csv": 900,           # ancient         -> ARCHIVE
}

def tier_for(age_days):              # the lifecycle policy as a simple function
    if age_days <= 7:    return "HOT (instant, priciest)"     # last week
    if age_days <= 30:   return "COOL (seconds, cheaper)"     # last month
    if age_days <= 180:  return "COLD (minutes, cheap)"       # last 6 months
    return "ARCHIVE (hours to restore, cheapest)"             # older than that

for name, age in objects_age.items():        # apply the policy to every object
    print(f"{name:22} age={age:>3}d -> {tier_for(age)}")  # show chosen tier
# Trade-off: archive is cheapest to store but slow + may cost a retrieval fee.


live_dashboard.csv     age=  0d -> HOT (instant, priciest)
last_month_report.csv  age= 20d -> COOL (seconds, cheaper)
q1_logs.csv            age=120d -> COLD (minutes, cheap)
2019_taxes.csv         age=900d -> ARCHIVE (hours to restore, cheapest)


*Your turn:* change `q1_logs.csv` age to `3` and re-run — watch its tier
jump back to HOT.

## 3 · Data lake vs data warehouse

- **Data lake** — store *raw* data as-is; apply structure later
  (**schema-on-read**). Flexible, cheap, great for exploration/ML.
- **Data warehouse** — clean, modelled tables; structure enforced up front
  (**schema-on-write**). Fast, governed SQL analytics.


In [24]:
import pandas as pd # Import pandas for better display of dataframes, if needed later

# ----- DATA LAKE: dump messy raw records (any shape allowed) -----
raw_lake = [                         # a list of dicts with INCONSISTENT fields
    {"event": "click", "user": "amy", "ts": 1, "created_at": "2023-01-01"}, # added 'created_at'
    {"event": "view",  "userId": 7,  "ts": 2},           # different key: 'userId'
    {"event": "buy",   "user": "bob", "ts": 3, "amt": 9} # extra field 'amt'
]
print("Lake stores anything, no fixed schema. Count:", len(raw_lake))

# Schema-on-READ: WE decide the structure only now, while reading.
for r in raw_lake:                                   # loop over raw records
    user = r.get("user") or r.get("userId")          # reconcile the two key names
    created_at = r.get("created_at", "N/A")          # get 'created_at' if it exists
    print(f"read -> {r["event"]} by {user} (created_at: {created_at})") # impose structure on the fly

Lake stores anything, no fixed schema. Count: 3
read -> click by amy (created_at: 2023-01-01)
read -> view by 7 (created_at: N/A)
read -> buy by bob (created_at: N/A)


In [7]:
# ----- DATA WAREHOUSE: enforce a clean schema up front (schema-on-write) -----
warehouse = []                       # will hold only well-shaped rows
def insert_event(event, user, ts):   # the function ENFORCES the schema
    assert isinstance(event, str), "event must be text"   # reject bad types
    assert isinstance(ts, int), "ts must be an integer"   # structure up front
    warehouse.append({"event": event, "user": str(user), "ts": ts})  # clean row

insert_event("click", "amy", 1)      # valid -> accepted
insert_event("view", 7, 2)           # 'user' coerced to text -> clean & uniform
print("Warehouse rows (uniform shape):", warehouse)

try:                                 # show the warehouse REJECTING bad data
    insert_event("buy", "bob", "three")   # ts is text, not int -> should fail
except AssertionError as e:          # catch the rejection
    print("Rejected by schema-on-write:", e)


Warehouse rows (uniform shape): [{'event': 'click', 'user': 'amy', 'ts': 1}, {'event': 'view', 'user': '7', 'ts': 2}]
Rejected by schema-on-write: ts must be an integer


*Your turn:* add a record to `raw_lake` with a brand-new field. The lake
accepts it instantly; the warehouse would need a schema change first.

## 4 · Relational database (SQL) with `sqlite3`

Relational = data in **tables** of rows & columns, linked by **keys**, queried
with **SQL**. `sqlite3` is a real SQL engine built into Python.

We make two tables — `customers` and `orders` — and **join** them.


In [8]:
con = sqlite3.connect(":memory:")    # create a throwaway in-memory database
cur = con.cursor()                   # a cursor runs SQL commands

cur.execute('''CREATE TABLE customers (   -- define the customers table
    id   INTEGER PRIMARY KEY,              -- unique id (the key)
    name TEXT NOT NULL                     -- a required name column
)''')

cur.execute('''CREATE TABLE orders (      -- define the orders table
    id          INTEGER PRIMARY KEY,       -- unique order id
    customer_id INTEGER,                   -- FOREIGN key -> customers.id
    amount      REAL                       -- order total (a number)
)''')

# Insert a little synthetic data using safe parameter placeholders (?)
cur.executemany("INSERT INTO customers VALUES (?,?)",
                [(1,"Amy"), (2,"Bob"), (3,"Cara")])      # 3 customers
cur.executemany("INSERT INTO orders VALUES (?,?,?)",
                [(10,1,50.0),(11,1,20.0),(12,2,99.0)])   # 3 orders
con.commit()                         # save (commit) the inserts
print("Tables created and rows inserted.")


Tables created and rows inserted.


In [9]:
# A JOIN combines rows from both tables using the matching key.
rows = cur.execute('''
    SELECT customers.name, orders.amount     -- pick name + amount
    FROM orders                              -- start from orders
    JOIN customers                           -- attach customers...
      ON orders.customer_id = customers.id   -- ...where the keys match
    WHERE orders.amount > 40                 -- NEW: only orders above 40
    ORDER BY orders.amount DESC              -- biggest order first
''').fetchall()                              # fetch every result row

for name, amount in rows:            # loop over the joined results
    print(f"{name} spent {amount}")  # show who spent what

# An aggregate query: total spend per customer (GROUP BY).
print("--- totals ---")
for name, total in cur.execute('''
    SELECT customers.name, SUM(orders.amount)   -- sum amounts...
    FROM orders JOIN customers
      ON orders.customer_id = customers.id
    GROUP BY customers.name                      -- ...one row per customer
''').fetchall():
    print(name, "total:", total)

Bob spent 99.0
Amy spent 50.0
Amy spent 20.0
--- totals ---
Amy total: 70.0
Bob total: 99.0


*Your turn:* give Cara an order and re-run the totals query.

## 5 · ACID — Atomicity in action (all-or-nothing)

ACID = Atomicity, Consistency, Isolation, Durability. We'll *see* **Atomicity**:
a money transfer must move funds out of one account AND into another — or do
**nothing at all** if something goes wrong. No money is ever lost or duplicated.


In [10]:
cur.execute("CREATE TABLE accounts (name TEXT, balance REAL)")  # new table
cur.executemany("INSERT INTO accounts VALUES (?,?)",
                [("Amy",100.0),("Bob",100.0)])   # both start with 100
con.commit()                                       # save starting balances

def show():                                        # tiny helper to print balances
    for n,b in cur.execute("SELECT name,balance FROM accounts"):
        print(" ", n, b)

print("Before:"); show()                           # show starting state


Before:
  Amy 100.0
  Bob 100.0


In [11]:
# Try a transfer that FAILS halfway. Atomicity must roll everything back.
try:
    cur.execute("BEGIN")                           # start a transaction
    cur.execute("UPDATE accounts SET balance=balance-30 WHERE name='Amy'")  # take 30
    raise ValueError("network glitch before crediting Bob!")  # simulate a crash
    cur.execute("UPDATE accounts SET balance=balance+30 WHERE name='Bob'")  # never runs
    con.commit()                                   # never reached
except ValueError as e:                            # the failure is caught here
    con.rollback()                                 # UNDO the partial change
    print("Transaction failed:", e, "-> rolled back")

print("After failed transfer (unchanged!):"); show()   # Amy still has 100


Transaction failed: network glitch before crediting Bob! -> rolled back
After failed transfer (unchanged!):
  Amy 100.0
  Bob 100.0


In [12]:
# Now a transfer that SUCCEEDS: both updates commit together.
cur.execute("BEGIN")                               # start transaction
cur.execute("UPDATE accounts SET balance=balance-30 WHERE name='Amy'")  # debit
cur.execute("UPDATE accounts SET balance=balance+30 WHERE name='Bob'")  # credit
con.commit()                                       # commit BOTH atomically
print("After successful transfer:"); show()        # Amy 70, Bob 130


After successful transfer:
  Amy 70.0
  Bob 130.0


**Takeaway:** with Atomicity the transfer is all-or-nothing — exactly why a
bank never loses or duplicates money mid-transaction.

## 6 · NoSQL families: key-value & document

NoSQL = non-relational stores built for flexible schemas and scale. Two common
families, modelled in Python:
- **Key-Value** (e.g. Redis) — fast lookups by key; perfect for caches/sessions.
- **Document** (e.g. MongoDB) — JSON-like records where each can differ.


In [13]:
# ---------- KEY-VALUE store (just a dict): O(1) lookup by key ----------
kv = {}                                   # the whole store is one dict
kv["session:amy"] = "logged_in"           # set a value by its key
kv["session:bob"] = "logged_in"           # another session
print("Lookup by key:", kv["session:amy"])# instant retrieval by exact key
print("Has bob?", "session:bob" in kv)    # membership check is also instant
# You can ONLY find things by key here — no rich queries. That's the trade-off.


Lookup by key: logged_in
Has bob? True


In [14]:
# ---------- DOCUMENT store (list of JSON-like dicts, flexible schema) ----------
docs = [                                   # each document can have DIFFERENT fields
    {"_id": 1, "name": "Amy", "tags": ["vip"]},          # has a 'tags' list
    {"_id": 2, "name": "Bob"},                           # no 'tags' at all
    {"_id": 3, "name": "Cara", "tags": ["new"], "age": 30},  # extra 'age'
]

# A simple query: find documents that have the tag 'vip'.
vips = [d for d in docs if "vip" in d.get("tags", [])]   # .get avoids KeyError
print("VIP docs:", json.dumps(vips))       # pretty-print as JSON text

# Adding a field to ONE document needs no migration (flexible schema).
docs[1]["age"] = 25                         # Bob gains an 'age' field on the fly
print("Bob now:", docs[1])


VIP docs: [{"_id": 1, "name": "Amy", "tags": ["vip"]}]
Bob now: {'_id': 2, 'name': 'Bob', 'age': 25}


*Your turn:* add a new document with a field none of the others have. It
just works — no `ALTER TABLE` required.

## 7 · SQL vs NoSQL — the schema difference, side by side

The same change (add a new field to *some* records) is cheap in NoSQL but needs
a schema migration in SQL.


In [15]:
# In SQL you must change the table definition for everyone (migration):
cur.execute("CREATE TABLE users_sql (id INTEGER, name TEXT)")  # rigid schema
cur.execute("INSERT INTO users_sql VALUES (1,'Amy')")          # add a row
# To store a 'phone' for only some users we must alter the WHOLE table:
cur.execute("ALTER TABLE users_sql ADD COLUMN phone TEXT")     # migration step
cur.execute("UPDATE users_sql SET phone='12345' WHERE id=1")   # then fill it
print("SQL row after migration:", cur.execute("SELECT * FROM users_sql").fetchone())

# In NoSQL (document) you just add the field to the records that need it:
users_nosql = [{"id":1,"name":"Amy","phone":"12345"},  # this one has a phone
               {"id":2,"name":"Bob"}]                   # this one simply doesn't
print("NoSQL records (mixed shapes):", users_nosql)
# Rule of thumb: SQL for integrity & rich queries; NoSQL for flexibility & scale.


SQL row after migration: (1, 'Amy', '12345')
NoSQL records (mixed shapes): [{'id': 1, 'name': 'Amy', 'phone': '12345'}, {'id': 2, 'name': 'Bob'}]


## 8 · Replication & high availability

Keep synchronized **copies**: the **primary** takes writes; **replicas** serve
reads and stand by for failover. We also show why **replication ≠ backup**.


In [16]:
class Node:                          # a tiny stand-in for one database server
    def __init__(self, name):        # each node has a name and its own data dict
        self.name = name             # e.g. 'primary'
        self.data = {}               # key -> value storage

primary  = Node("primary")           # the only node that accepts WRITES
replicaA = Node("replicaA")          # read replica / failover standby
replicaB = Node("replicaB")          # another replica
replicas = [replicaA, replicaB]      # group the replicas together

def write(key, value):               # writes ALWAYS go to the primary first
    primary.data[key] = value        # apply the write on primary
    for r in replicas:               # then copy ('replicate') to each replica
        r.data[key] = value          # replica now mirrors the primary

write("user:1", "Amy")               # perform a replicated write
write("user:2", "Bob")               # another one
print("Primary:", primary.data)      # primary holds the data
print("ReplicaA (reads):", replicaA.data)  # replica mirrors it -> can serve reads


Primary: {'user:1': 'Amy', 'user:2': 'Bob'}
ReplicaA (reads): {'user:1': 'Amy', 'user:2': 'Bob'}


In [17]:
# Failover: if the primary dies, promote a replica to be the new primary.
print("Primary alive? simulate a crash...")
primary = replicaA                   # 'promote' replicaA: it becomes primary
print("New primary is now replicaA with data:", primary.data)  # no data lost

# Replication is NOT a backup: a bad delete replicates everywhere instantly.
def delete_everywhere(key):          # simulate an accidental destructive op
    primary.data.pop(key, None)      # remove from primary
    for r in [replicaB]:             # ...and it propagates to replicas too
        r.data.pop(key, None)
delete_everywhere("user:1")          # oops, deleted on purpose-by-accident
print("After bad delete -> gone from replica too:", replicaB.data)
# Lesson: you still need real BACKUPS (next section) to recover from mistakes.


Primary alive? simulate a crash...
New primary is now replicaA with data: {'user:1': 'Amy', 'user:2': 'Bob'}
After bad delete -> gone from replica too: {'user:2': 'Bob'}


## 9 · Partitioning & sharding

**Sharding** splits one big table across machines by a **shard key**, so each
shard holds a slice. We shard users by the first letter of their name.


In [18]:
shards = {"A-F": {}, "G-M": {}, "N-S": {}, "T-Z": {}}  # 4 empty shards

def shard_for(name):                 # decide which shard a name belongs to
    first = name[0].upper()          # look at the first letter
    if first <= "F": return "A-F"    # A..F go to shard 1
    if first <= "M": return "G-M"    # G..M go to shard 2
    if first <= "S": return "N-S"    # N..S go to shard 3
    return "T-Z"                     # T..Z go to shard 4

for name in ["Amy","Bob","Liam","Nora","Zoe","Cara"]:  # synthetic users
    target = shard_for(name)         # compute the right shard
    shards[target][name] = "..."     # store the user in that shard
    print(f"{name:5} -> shard {target}")  # show the routing decision

print("Shard sizes:", {k: len(v) for k,v in shards.items()})  # rows per shard
# Watch out: if most names start with A-F, that shard becomes a 'hot' shard.


Amy   -> shard A-F
Bob   -> shard A-F
Liam  -> shard G-M
Nora  -> shard N-S
Zoe   -> shard T-Z
Cara  -> shard A-F
Shard sizes: {'A-F': 3, 'G-M': 1, 'N-S': 1, 'T-Z': 1}


*Your turn:* add several names starting with 'A'. See the `A-F` shard grow
unevenly — that's a **skewed shard key** creating a hot shard.

## 10 · Indexing — why reads get fast

Without an index the DB does a **full table scan** (check every row, O(n)). An
index (a sorted lookup) jumps near-directly to matches (≈ O(log n)). We time
both on synthetic data.


In [19]:
N = 200_000                                   # number of synthetic rows
con2 = sqlite3.connect(":memory:")            # a fresh in-memory DB
c2 = con2.cursor()
c2.execute("CREATE TABLE big (id INTEGER, val INTEGER)")  # no index yet
rows = [(i, random.randint(0, N)) for i in range(N)]      # fake (id, val) rows
c2.executemany("INSERT INTO big VALUES (?,?)", rows)      # bulk insert
con2.commit()
print("Inserted", N, "rows.")


Inserted 200000 rows.


In [20]:
target = rows[N//2][1]                # pick a value we know exists to search for

t0 = time.time()                       # start timer (no index = full scan)
c2.execute("SELECT COUNT(*) FROM big WHERE val=?", (target,)).fetchone()
no_index = time.time() - t0            # time taken without an index

c2.execute("CREATE INDEX idx_val ON big(val)")  # build an index on 'val'

t0 = time.time()                       # start timer again (with index)
c2.execute("SELECT COUNT(*) FROM big WHERE val=?", (target,)).fetchone()
with_index = time.time() - t0          # time taken with the index

print(f"Without index: {no_index*1000:7.2f} ms")   # slower (scans everything)
print(f"With index:    {with_index*1000:7.2f} ms")  # faster (jumps to matches)
print(f"Speed-up:      ~{no_index/max(with_index,1e-9):.0f}x faster")
# Trade-off: indexes speed reads but cost extra space and slow down writes.


Without index:   14.49 ms
With index:       0.28 ms
Speed-up:      ~52x faster


## 11 · CAP theorem (a tiny simulation)

During a **network partition** you can't have both perfect **Consistency** and
full **Availability** — you must trade one for the other. Below, two replicas
get cut off; we show the **CP** choice (refuse stale reads) vs the **AP** choice
(answer with possibly-stale data).


In [21]:
nodeA = {"x": 1}                      # both nodes start in agreement (x = 1)
nodeB = {"x": 1}
partitioned = True                    # the link between A and B is broken

nodeA["x"] = 2                        # a write lands on node A -> A now has x=2
# Because of the partition, B did NOT receive the update; B still thinks x=1.

def read_CP(node):                    # CP choice: prefer Consistency
    if partitioned:                   # if we might be stale...
        return "ERROR: refuse read to stay consistent"  # ...sacrifice Availability
    return node["x"]

def read_AP(node):                    # AP choice: prefer Availability
    return node["x"]                  # always answer, even if possibly stale

print("Reading node B during partition:")
print("  CP system:", read_CP(nodeB))   # refuses -> consistent but unavailable
print("  AP system:", read_AP(nodeB))   # returns stale 1 -> available but stale
print("Node A truly has x =", nodeA["x"])  # the latest value lives on A
# Real systems pick a side; many choose AP with 'eventual' consistency.


Reading node B during partition:
  CP system: ERROR: refuse read to stay consistent
  AP system: 1
Node A truly has x = 2


## 12 · Backups & durability — snapshot and restore

Replication protects against *hardware* failure; **backups** protect against
*mistakes* (a bad DELETE replicates too!). Here we snapshot a database file,
then "corrupt" the data, then **restore** from the snapshot.


In [22]:
db_path = os.path.join(WORK, "app.db")        # path to a real on-disk DB file
con3 = sqlite3.connect(db_path)               # create/open it
con3.execute("CREATE TABLE notes (text TEXT)")# one simple table
con3.executemany("INSERT INTO notes VALUES (?)",
                 [("keep me 1",),("keep me 2",)])  # important data
con3.commit(); con3.close()                   # save and close so we can copy the file

snapshot = os.path.join(WORK, "app_backup.db")  # where the backup will live
shutil.copy(db_path, snapshot)                  # the BACKUP = a point-in-time copy
print("Snapshot taken ->", os.path.basename(snapshot))


Snapshot taken -> app_backup.db


In [23]:
# Disaster: someone wipes the table (and imagine this replicated everywhere).
con3 = sqlite3.connect(db_path)               # reopen the live DB
con3.execute("DELETE FROM notes")             # accidental destructive command
con3.commit()
print("Live rows after bad delete:", con3.execute("SELECT COUNT(*) FROM notes").fetchone()[0])
con3.close()

# Restore: replace the live file with our snapshot copy.
shutil.copy(snapshot, db_path)                # overwrite live DB with the backup
con3 = sqlite3.connect(db_path)               # reopen the restored DB
restored = con3.execute("SELECT * FROM notes").fetchall()  # read the data back
print("Rows after restore from backup:", restored)         # data is back!
con3.close()
# RPO = how much data you can afford to lose; RTO = how fast you must recover.
# Golden rule: an UNTESTED backup is the same as NO backup — always test restores.


Live rows after bad delete: 0
Rows after restore from backup: [('keep me 1',), ('keep me 2',)]


## 13 · Wrap-up & mini-exercises

**You practically saw:**
- object vs block vs file storage, and storage **tiers** (hot → archive)
- **lake** (schema-on-read) vs **warehouse** (schema-on-write)
- **SQL** tables/joins and **ACID** atomicity (all-or-nothing transfer)
- **NoSQL** key-value & document stores, and the SQL-vs-NoSQL schema trade-off
- **replication** + failover, and why replication ≠ backup
- **sharding** by a shard key (and hot shards)
- **indexing** speed-up (timed), the **CAP** trade-off, and **backup/restore**

**Try these (small):**
1. Add a `created_at` field to only *some* lake records, then re-run section 3.
2. In section 4, write a SQL query for orders **above 40** using `WHERE`.
3. In section 9, add 6 names starting with 'B' and find the new hot shard.
4. In section 10, change `N` to 1,000,000 and compare the speed-up.

**Cleanup (optional):** run the next cell to delete the scratch folder.


In [ ]:
shutil.rmtree(WORK, ignore_errors=True)   # remove all files we created
print("Scratch folder removed. Notebook complete!")  # done
